<a id="top"></a>

# Lab L1404: build the shallow agent as a graph

```yaml
title: "Lab L1404: build the shallow agent as a graph"
keywords: langgraph, stategraph, shallow agent, node, conditional edge, back-edge, toolnode, add_messages, typed state, compile, invoke, lab
estimated duration: 35
```

> **Lesson:** L14. See [objectives.md](../../../../docs/origin/lesson_roadmaps/L14/objectives.md).
> **Solutions:** `L1404_lab_solutions.ipynb`.
> Runs **fully offline** -- a deterministic `StubChat` stands in for `ChatAnthropic`, and the REAL
> `ToolNode` runs the REAL `common/tools.py` tools. No API key needed. You wire the graph.

**You will:** model the L10 agent loop as a graph -- a typed state, an `agent` node, the prebuilt
`tools` node, a conditional edge, and the **back-edge** that makes it an agent (objective 2).

## Contents

- [1. Setup](#1-setup)
- [2. Problem 1 -- The typed state](#2-problem-1----the-typed-state)
- [3. Problem 2 -- The agent node and the router](#3-problem-2----the-agent-node-and-the-router)
- [4. Problem 3 -- Wire, compile, render](#4-problem-3----wire-compile-render)
- [5. Problem 4 -- Run the agent](#5-problem-4----run-the-agent)
- [6. Problem 5 -- What makes it an agent? (written)](#6-problem-5----what-makes-it-an-agent-written)
- [7. Self-check](#7-self-check)

## 1. Setup

Everything here is **given**: the shared tools wrapped for LangChain, the offline `StubChat`
(scripted `calculator` -> `lookup` -> answer), and a `model` stub. Run this cell first.

In [1]:
from typing import Annotated, TypedDict

from langchain_core.messages import AIMessage, BaseMessage, HumanMessage
from langchain_core.tools import StructuredTool
from langgraph.graph import END, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode

from fluffy_potato_curriculum.common import tools as T

# The SHARED tools (L10/L11), wrapped as LangChain tools -- the same functions, unchanged.
calculator = StructuredTool.from_function(T.calculator, name="calculator", description="arithmetic")
lookup = StructuredTool.from_function(T.lookup, name="lookup", description="city population")
flaky_fetch = StructuredTool.from_function(
    T.flaky_fetch, name="flaky_fetch", description="fetch url"
)
LC_TOOLS = [calculator, lookup, flaky_fetch]


class StubChat:
    """A tiny OFFLINE stand-in for ChatAnthropic bound with tools, so this lab is free, fast,
    and deterministic. It SCRIPTS the chaining task by counting how many AI turns have happened:
    turn 0 -> ask for `calculator`, turn 1 -> ask for `lookup`, then -> a final text answer.

    The graph wiring you practice is identical with a real model: swapping `StubChat()` for
    `ChatAnthropic(model=SONNET, api_key=require_anthropic_key())` is a one-line change, and the
    node code never changes (Problem 5). The lecture demo (L1403) uses the real client. The REAL
    `ToolNode` runs the REAL tools here -- only the *model* is faked.
    """

    def bind_tools(self, tools: list[object]) -> "StubChat":
        return self  # a real ChatAnthropic returns a bound copy; the stub needs no schemas

    def invoke(self, messages: list[BaseMessage]) -> AIMessage:
        ai_turns = sum(1 for m in messages if isinstance(m, AIMessage))
        if ai_turns == 0:
            return AIMessage(
                content="",
                tool_calls=[{"name": "calculator", "args": {"expression": "21*2"}, "id": "c1"}],
            )
        if ai_turns == 1:
            return AIMessage(
                content="", tool_calls=[{"name": "lookup", "args": {"city": "Tokyo"}, "id": "l1"}]
            )
        return AIMessage(content="21*2 = 42, and Tokyo's population is 37000000.")


model = StubChat()
TASK = "Use the calculator to compute 21 * 2, then tell me the population of Tokyo."

[↑ Back to top](#top)

## 2. Problem 1 -- The typed state

Define the **state** the agent threads through its nodes. It needs `messages` with the **append
reducer** (`Annotated[list[BaseMessage], add_messages]`) so each node *adds* to the history, and a
`step: int` counter (default overwrite reducer). Getting the `messages` reducer wrong reproduces the
most common L10 bug -- you'll see exactly how in L1405.

In [2]:
class AgentState(TypedDict):
    """The running message history (appended) plus one step counter (overwritten)."""

    messages: Annotated[
        list[BaseMessage], add_messages
    ]  # APPENDS -- preserves tool_use/result order
    step: int  # overwrites (default reducer)

[↑ Back to top](#top)

## 3. Problem 2 -- The agent node and the router

Write `agent_node` (call the model on `state["messages"]`, return the response to be appended plus
an incremented `step`) and `route` (the conditional edge's function: `"tools"` if the last message
asked for a tool, else `END`). The `tools` node is **given** -- the prebuilt `ToolNode`.

In [3]:
def agent_node(state: AgentState) -> dict[str, object]:
    """L10's 'call the model' step, as a node -- returns the response to be APPENDED."""
    bound = model.bind_tools(LC_TOOLS)
    response = bound.invoke(state["messages"])
    return {"messages": [response], "step": state["step"] + 1}


def route(state: AgentState) -> str:
    """L10's 'is there a tool_use?' branch, as a routing function on the conditional edge."""
    last = state["messages"][-1]
    if isinstance(last, AIMessage) and last.tool_calls:
        return "tools"
    return END


# GIVEN: the prebuilt tool node (L10's dispatch step, packaged).
tool_node = ToolNode(LC_TOOLS, handle_tool_errors=True)

[↑ Back to top](#top)

## 4. Problem 3 -- Wire, compile, render

Wire the graph: entry at `agent`, a **conditional edge** out of `agent` (`route` -> `tools` or
`END`), and the **back-edge** `tools -> agent`. `compile()` to `agent_app` and print the diagram.
Notice the render needs no model -- the control flow is **data**.

In [4]:
builder = StateGraph(AgentState)
builder.add_node("agent", agent_node)
builder.add_node("tools", tool_node)
builder.set_entry_point("agent")
builder.add_conditional_edges("agent", route, {"tools": "tools", END: END})
builder.add_edge("tools", "agent")  # the back-edge: the only thing L14 adds to a workflow
agent_app = builder.compile()

# The render needs no model at all -- the control flow is just DATA.
print(agent_app.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	agent(agent)
	tools(tools)
	__end__([<p>__end__</p>]):::last
	__start__ --> agent;
	agent -.-> __end__;
	agent -.-> tools;
	tools --> agent;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



[↑ Back to top](#top)

## 5. Problem 4 -- Run the agent

`invoke()` the graph on `TASK` and print the **tool path**, the **step** count, and the **final
text**. The path should be `['calculator', 'lookup']` -- the same sequence the L10 loop produced.
(The stub's text is fixed; a real model's *wording* would vary, but the *path* is the equivalence
check.)

In [5]:
result = agent_app.invoke({"messages": [HumanMessage(TASK)], "step": 0})
path = [c["name"] for m in result["messages"] if isinstance(m, AIMessage) for c in m.tool_calls]
print("tool path :", path)  # ['calculator', 'lookup'] -- the SAME sequence the L10 loop made
print("steps     :", result["step"])
print("final text:", result["messages"][-1].text)

tool path : ['calculator', 'lookup']
steps     : 3
final text: 21*2 = 42, and Tokyo's population is 37000000.


[↑ Back to top](#top)

## 6. Problem 5 -- What makes it an agent? (written)

_Double-click to edit._

**Q1.** Which single edge turns this graph from a *workflow* (L04) into an *agent*? Name it.

The **back-edge `tools -> agent`**. It loops control back to the model after tools run, so the
*model* decides whether to call another tool or stop -- the model now drives the path. Without it,
every path runs forward to `END` and the graph is an acyclic workflow.

**Q2.** ...Why does none of the node code change?

Because the node only calls `model.bind_tools(...).invoke(messages)` -- and `StubChat` and
`ChatAnthropic` both expose that same `bind_tools(...).invoke(...)` interface. The node depends on
the *interface*, not the concrete client, so swapping the client is a one-line change at
construction; the graph and nodes are untouched.

[↑ Back to top](#top)

## 7. Self-check

Compare against `L1404_lab_solutions.ipynb`. You're done when: `AgentState` has `messages`
(`add_messages`) + `step`; `agent_node` returns a partial update and `route` returns `"tools"`/`END`;
`agent_app` compiles and `draw_mermaid()` shows the cycle `agent -> tools -> agent` with an edge to
`END`; invoking yields `['calculator', 'lookup']`; and you can name the **back-edge** as the one
thing that makes it an agent.

[↑ Back to top](#top)